In [11]:
import csv
import json
import logging
import random
import re
import time
from datetime import datetime

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ============================================================
# CONFIG
# ============================================================

BASE_URL = "https://api.hh.ru"
SEARCH_FIELD = "name,description"

# Только нужные города
SELECTED_AREAS = {
    "Алматы": 160,
    "Астана": 167,
}

# В detail-check допускаем только Алматы и Астану
ALLOWED_AREA_IDS = {160, 167}

HEADERS = {
    "User-Agent": "KZ-IT-Jobs-Parser/1.0 (Educational project)",
    "HH-User-Agent": "KZ-IT-Jobs-Parser/1.0 (Educational project)",
}

OUTPUT_CSV = f"hh_almaty_astana_it_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
OUTPUT_JSON = f"hh_almaty_astana_it_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

IT_VACANCIES = [
    # Data & Analytics
    "Data Analyst", "Аналитик данных", "Data Scientist", "Data Science",
    "Data Engineer", "Инженер данных", "BI Developer", "BI-аналитик",
    "Business Intelligence", "Web Analyst", "Веб-аналитик",

    # Machine Learning & AI
    "Machine Learning Engineer", "ML Engineer", "ML-инженер", "MLOps",
    "Deep Learning", "Computer Vision", "NLP Engineer", "AI Engineer",
    "Инженер по искусственному интеллекту",

    # Backend
    "Backend Developer", "Backend-разработчик", "Python Developer",
    "Python-разработчик", "Java Developer", "Java-разработчик",
    "Golang Developer", "Go Developer", "Node.js Developer",
    ".NET Developer", "C# Developer", "PHP Developer", "Kotlin Developer",

    # Frontend
    "Frontend Developer", "Frontend-разработчик", "React Developer",
    "Vue Developer", "Angular Developer", "JavaScript Developer",
    "TypeScript Developer",

    # Mobile
    "iOS Developer", "iOS-разработчик", "Android Developer",
    "Android-разработчик", "Flutter Developer", "React Native Developer",
    "Мобильный разработчик",

    # Full Stack
    "Full Stack Developer", "Fullstack Developer", "Фулстек разработчик",

    # DevOps & Infrastructure
    "DevOps Engineer", "DevOps-инженер", "SRE Engineer", "Cloud Engineer",
    "Kubernetes Engineer", "Системный администратор",
    "System Administrator", "Network Engineer", "Сетевой инженер",

    # QA & Testing
    "QA Engineer", "QA-инженер", "Тестировщик", "QA Automation",
    "Автоматизатор тестирования", "Manual QA",

    # Security
    "Information Security", "Информационная безопасность",
    "Cybersecurity Engineer", "Специалист по кибербезопасности",
    "Penetration Tester", "Security Analyst",

    # Analytics & Management
    "Системный аналитик", "System Analyst", "Бизнес-аналитик",
    "Business Analyst", "Product Manager", "Продукт-менеджер",
    "Product Owner", "Project Manager IT", "IT Project Manager",
    "Scrum Master", "Agile Coach",

    # Database
    "Database Administrator", "DBA", "Администратор баз данных",
    "SQL Developer",

    # General IT
    "Software Engineer", "Программист", "Разработчик ПО",
    "IT Specialist", "IT-специалист", "1C Developer",
    "1С разработчик", "ERP Developer", "SAP Developer",
]

# ============================================================
# HTTP / SESSION
# ============================================================

def build_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(HEADERS)

    retries = Retry(
        total=4,
        connect=4,
        read=4,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def polite_sleep(a: float = 0.7, b: float = 1.4) -> None:
    time.sleep(random.uniform(a, b))


# ============================================================
# HELPERS
# ============================================================

def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_salary(salary: dict | None) -> tuple:
    if not salary:
        return None, None, None, None
    return (
        salary.get("from"),
        salary.get("to"),
        salary.get("currency"),
        salary.get("gross"),
    )


# ============================================================
# API CALLS
# ============================================================

def get_vacancies_page(
    session: requests.Session,
    query: str,
    area_id: int,
    page: int = 0,
    search_field: str = SEARCH_FIELD,
) -> dict:
    url = f"{BASE_URL}/vacancies"

    params = [
        ("text", query),
        ("area", area_id),
        ("per_page", 100),
        ("page", page),
        ("order_by", "publication_time"),
    ]

    for field in search_field.split(","):
        params.append(("search_field", field.strip()))

    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


def get_vacancy_detail(session: requests.Session, vacancy_id: str) -> dict:
    url = f"{BASE_URL}/vacancies/{vacancy_id}"
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


# ============================================================
# PARSING
# ============================================================

def parse_vacancy(item: dict, search_city: str) -> dict:
    salary_from, salary_to, currency, salary_gross = parse_salary(item.get("salary"))

    return {
        "id": item.get("id"),
        "title": item.get("name"),

        "search_city": search_city,
        "vacancy_city": (item.get("area") or {}).get("name", ""),

        "company": (item.get("employer") or {}).get("name", ""),
        "company_id": (item.get("employer") or {}).get("id", ""),
        "experience": (item.get("experience") or {}).get("name", ""),
        "employment": (item.get("employment") or {}).get("name", ""),
        "schedule": (item.get("schedule") or {}).get("name", ""),

        "salary_from": salary_from,
        "salary_to": salary_to,
        "currency": currency,
        "salary_gross": salary_gross,

        "requirement": clean_text((item.get("snippet") or {}).get("requirement", "")),
        "responsibility": clean_text((item.get("snippet") or {}).get("responsibility", "")),

        "skills": "",
        "description": "",
        "work_format": "",
        "working_hours": "",

        # исправленная логика
        "vacancy_type": "",
        "internship": None,

        "url": item.get("alternate_url"),
        "published_at": item.get("published_at"),
    }


def enrich_vacancy(
    session: requests.Session,
    vacancy: dict,
    retries: int = 3,
) -> dict:
    """
    Получить полное описание + навыки из детального запроса.
    """
    for attempt in range(retries):
        try:
            detail = get_vacancy_detail(session, vacancy["id"])

            detail_area = detail.get("area") or {}
            area_id = detail_area.get("id")

            if area_id is not None:
                try:
                    if int(area_id) not in ALLOWED_AREA_IDS:
                        vacancy["_skip"] = True
                        return vacancy
                except (TypeError, ValueError):
                    pass

            vacancy["vacancy_city"] = detail_area.get("name", vacancy.get("vacancy_city", ""))

            vacancy["description"] = clean_text(detail.get("description", ""))

            skills = [
                skill.get("name", "")
                for skill in detail.get("key_skills", [])
                if skill.get("name")
            ]
            vacancy["skills"] = ", ".join(skills)

            salary_from, salary_to, currency, salary_gross = parse_salary(detail.get("salary"))
            if any(v is not None for v in [salary_from, salary_to, currency, salary_gross]):
                vacancy["salary_from"] = salary_from
                vacancy["salary_to"] = salary_to
                vacancy["currency"] = currency
                vacancy["salary_gross"] = salary_gross

            vacancy["schedule"] = (detail.get("schedule") or {}).get("name", vacancy.get("schedule", ""))
            vacancy["employment"] = (detail.get("employment") or {}).get("name", vacancy.get("employment", ""))
            vacancy["experience"] = (detail.get("experience") or {}).get("name", vacancy.get("experience", ""))

            vacancy["work_format"] = ", ".join(
                x.get("name", "")
                for x in detail.get("working_time_modes", [])
                if x.get("name")
            )

            vacancy["working_hours"] = ", ".join(
                x.get("name", "")
                for x in detail.get("working_hours", [])
                if x.get("name")
            )

            # type и internship — разные поля
            vacancy["vacancy_type"] = (detail.get("type") or {}).get("name", "")
            vacancy["internship"] = detail.get("internship")

            polite_sleep(0.9, 1.8)
            return vacancy

        except requests.exceptions.ConnectTimeout:
            wait = (attempt + 1) * 4
            logging.warning(
                f"Timeout for vacancy {vacancy['id']}, attempt {attempt + 1}/{retries}, waiting {wait}s"
            )
            time.sleep(wait)
        except requests.exceptions.ConnectionError:
            logging.warning(f"Connection error for vacancy {vacancy['id']}, skipping")
            break
        except requests.HTTPError as e:
            logging.warning(f"HTTP error for vacancy {vacancy['id']}: {e}")
            break
        except Exception as e:
            logging.warning(f"Unexpected error for vacancy {vacancy['id']}: {e}")
            break

    return vacancy


# ============================================================
# MAIN PARSER
# ============================================================

def parse_all_vacancies(
    session: requests.Session,
    areas: dict,
    queries: list,
    fetch_skills: bool = True,
    search_field: str = SEARCH_FIELD,
) -> list:
    all_vacancies = []

    # дедупликация по id + search_city
    seen_keys = set()

    for city_name, area_id in areas.items():
        logging.info(f"=== Parsing city: {city_name} ===")

        for query in queries:
            logging.info(f"  Query [{search_field}]: '{query}'")
            page = 0

            while True:
                try:
                    data = get_vacancies_page(
                        session=session,
                        query=query,
                        area_id=area_id,
                        page=page,
                        search_field=search_field,
                    )

                    items = data.get("items", [])
                    if not items:
                        break

                    new_count = 0

                    for item in items:
                        vacancy_id = item.get("id")
                        if not vacancy_id:
                            continue

                        dedup_key = (vacancy_id, city_name)
                        if dedup_key in seen_keys:
                            continue
                        seen_keys.add(dedup_key)

                        vacancy = parse_vacancy(item, city_name)

                        if fetch_skills:
                            vacancy = enrich_vacancy(session, vacancy)

                        if vacancy.get("_skip"):
                            continue

                        all_vacancies.append(vacancy)
                        new_count += 1

                    total_pages = data.get("pages", 1)
                    logging.info(
                        f"    Page {page + 1}/{total_pages} | new: {new_count} | total: {len(all_vacancies)}"
                    )

                    if page >= total_pages - 1:
                        break

                    page += 1
                    polite_sleep()

                except requests.HTTPError as e:
                    logging.error(f"HTTP error for query '{query}' in {city_name}: {e}")
                    break
                except Exception as e:
                    logging.error(f"Unexpected error in city={city_name}, query='{query}': {e}")
                    break

    logging.info(f"✅ Done. Total rows: {len(all_vacancies)}")
    return all_vacancies


# ============================================================
# SAVE
# ============================================================

def save_to_csv(vacancies: list, filename: str = OUTPUT_CSV) -> str:
    if not vacancies:
        logging.warning("No data to save to CSV.")
        return ""

    fieldnames = [
    "id",
    "title",
    "search_city",
    "vacancy_city",
    "company",
    "company_id",
    "experience",
    "employment",
    "schedule",
    "salary_from",
    "salary_to",
    "currency",
    "salary_gross",
    "work_format",
    "working_hours",
    "vacancy_type",
    "internship",
    "skills",
    "requirement",
    "responsibility",
    "description",
    "url",
    "published_at",
    ]

    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(vacancies)

    logging.info(f"CSV saved: {filename}")
    return filename


def save_to_json(vacancies: list, filename: str = OUTPUT_JSON) -> str:
    if not vacancies:
        logging.warning("No data to save to JSON.")
        return ""

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(vacancies, f, ensure_ascii=False, indent=2)

    logging.info(f"JSON saved: {filename}")
    return filename


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    session = build_session()

    logging.info("🚀 Starting hh.kz parser for Almaty + Astana IT vacancies")
    logging.info(f"Cities: {', '.join(SELECTED_AREAS.keys())}")
    logging.info(f"Queries: {len(IT_VACANCIES)}")

    vacancies = parse_all_vacancies(
        session=session,
        areas=SELECTED_AREAS,
        queries=IT_VACANCIES,
        fetch_skills=True,
        search_field=SEARCH_FIELD,
    )

    # финальная защита от дублей
    unique_rows = []
    seen_final = set()

    for v in vacancies:
        key = (v["id"], v["search_city"])
        if key in seen_final:
            continue
        seen_final.add(key)
        unique_rows.append(v)

    save_to_csv(unique_rows, OUTPUT_CSV)
    save_to_json(unique_rows, OUTPUT_JSON)

    if unique_rows:
        v = unique_rows[0]
        print("\n=== Example vacancy ===")
        print(f"Title:         {v['title']}")
        print(f"Search city:   {v['search_city']}")
        print(f"Vacancy city:  {v['vacancy_city']}")
        print(f"Company:       {v['company']}")
        print(f"Salary:        {v['salary_from']} – {v['salary_to']} {v['currency']}")
        print(f"Experience:    {v['experience']}")
        print(f"Employment:    {v['employment']}")
        print(f"Skills:        {v['skills']}")
        print(f"URL:           {v['url']}")

2026-04-14 01:45:47,524 - INFO - 🚀 Starting hh.kz parser for Almaty + Astana IT vacancies
2026-04-14 01:45:47,525 - INFO - Cities: Алматы, Астана
2026-04-14 01:45:47,526 - INFO - Queries: 95
2026-04-14 01:45:47,526 - INFO - === Parsing city: Алматы ===
2026-04-14 01:45:47,526 - INFO -   Query [name,description]: 'Data Analyst'
2026-04-14 01:48:22,075 - INFO -     Page 1/2 | new: 100 | total: 100
2026-04-14 01:49:15,914 - INFO -     Page 2/2 | new: 31 | total: 131
2026-04-14 01:49:15,920 - INFO -   Query [name,description]: 'Аналитик данных'
2026-04-14 01:51:42,685 - INFO -     Page 1/6 | new: 94 | total: 225
2026-04-14 01:54:10,483 - INFO -     Page 2/6 | new: 92 | total: 317
2026-04-14 01:56:25,935 - INFO -     Page 3/6 | new: 86 | total: 403
2026-04-14 01:58:36,363 - INFO -     Page 4/6 | new: 85 | total: 488
2026-04-14 02:01:44,238 - WARNING - Retrying (Retry(total=3, connect=4, read=3, redirect=None, status=None)) after connection broken by 'RemoteDisconnected('Remote end closed co


=== Example vacancy ===
Title:         Client Operations & Solutions Specialist
Search city:   Алматы
Vacancy city:  Алматы
Company:       OUTSOURCE PRO
Salary:        500 – 2000 USD
Experience:    Нет опыта
Employment:    Полная занятость
Skills:        Английский язык, Деловая коммуникация, Аналитическое мышление, Работа с большим объемом информации
URL:           https://hh.ru/vacancy/131319207
